# π0.5-LIBERO × PARC 2026 提出環境

`feature/experiment` の `MyPolicy` をGPUでスモークし、公開4タスクで評価して、外部通信なしで動く `pi05_submission.zip` を作るノートブックです。既存のSmolVLAノートブックとは環境を分離します。

- ランタイム: GPU（L4 / A100推奨）
- PaliGemmaの利用条件にHugging Face上で同意してください
- W&Bは使用せず、ノートブック内でオフライン無効化します
- このノートブックは既存のLIBERO fine-tuned checkpointの導入用です。追加学習は含みません


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

REPO_URL = "https://github.com/KosukeKomeya/PARC2026_pre.git"
BRANCH = "feature/experiment"
REPO_DIR = Path("/content/PARC2026_pre")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
%cd /content/PARC2026_pre


In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ColabのランタイムをGPUに変更してください")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("VRAM GiB:", round(props.total_memory / 2**30, 1))
subprocess.run(["nvidia-smi"], check=True)


## Hugging Faceへログイン

`google/paligemma-3b-pt-224` のページで利用条件へ同意してから実行します。トークンをコードやGitHubへ書かないでください。


In [ ]:
%pip install -q "huggingface-hub>=0.34.2,<0.36.0"
from huggingface_hub import notebook_login
notebook_login()


## 1. 固定バージョンの環境と重みを準備

Python 3.10の専用venvを作り、固定コミットのLeRobot/Transformers、固定revisionのπ0.5-LIBERO重み、tokenizerを取得します。初回は時間がかかります。


In [ ]:
!python examples/pi05_parc_colab_setup.py


## 2. 実モデルでスモークテスト

モデルのロード、最初の重い推論、action queueから返す軽い推論を計測します。重い推論が10秒以上なら提出前にGPU・推論step数を見直してください。


In [ ]:
!python examples/pi05_parc_colab_setup.py --skip-download --smoke


## 3. オフライン提出ZIPを作成

チェックポイント、tokenizer、固定ソース、`policy_server.py`、`requirements.txt`をルート直下へまとめます。モデル重みは再圧縮せずZIP64で格納します。


In [ ]:
!python examples/pi05_parc_colab_setup.py --skip-download --build-submission


## 4. 公開4タスクで評価

PARC配布キットの `libero_t1`（公開4タスク）を、提出時と同じHTTPインターフェースと10秒タイムアウトで評価します。評価用のCPU環境は `venv/`、π0.5サーバーは既存の `/content/pi05_py310` GPU環境で動かし、PyTorch環境を分離します。初回の評価環境準備には10〜20分かかります。

既定はクイック確認用の1 episode/task（合計4エピソード）です。より安定した成功率を確認する場合は `EVAL_EPISODES_PER_TASK = 20` に変更してください。


In [ ]:
import csv
import os

PUBLIC_TASKS_CSV = REPO_DIR / "compe" / "t1" / "T1_TASKS.csv"
with PUBLIC_TASKS_CSV.open(encoding="utf-8", newline="") as handle:
    public_task_rows = list(csv.DictReader(handle))

PUBLIC_TASK_IDS = [row["task_id"] for row in public_task_rows]
if len(PUBLIC_TASK_IDS) != 4 or len(set(PUBLIC_TASK_IDS)) != 4:
    raise RuntimeError(f"公開タスクは4件である必要があります: {PUBLIC_TASK_IDS}")

EVAL_EPISODES_PER_TASK = 1  # 正式寄りの評価では20へ変更
EVAL_MAX_STEPS = 600
EVAL_SEED = 42
RECORD_VIDEO = True
VIDEOS_PER_TASK = 1  # 20 episodesでも各タスク先頭1本だけ保存

if EVAL_EPISODES_PER_TASK < 1 or EVAL_MAX_STEPS < 1:
    raise ValueError("episode数とmax stepsは1以上にしてください")

print(f"公開タスク: {len(PUBLIC_TASK_IDS)}件")
for index, row in enumerate(public_task_rows, 1):
    print(f"  {index}. {row['instruction']} ({row['task_id']})")
print(f"評価量: {len(PUBLIC_TASK_IDS) * EVAL_EPISODES_PER_TASK} episodes")


### 評価環境を準備

MuJoCoの描画ライブラリ、評価専用venv、LIBERO-plus、公開タスクassetsを準備します。`setup.sh` は `~/.libero/config.yaml` を評価環境向けに設定し、既存ファイルがあれば `.bak` へ退避します。


In [ ]:
system_packages = [
    "libosmesa6", "libgl1", "libglfw3", "libglew2.2",
    "libegl1", "libsm6", "libxext6", "libxrender1",
    "libglib2.0-0", "libmagickwand-dev", "unzip",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *system_packages],
    check=True,
)

PI05_PYTHON = Path("/content/pi05_py310/bin/python")
if not PI05_PYTHON.is_file():
    raise FileNotFoundError("先に手順1のπ0.5環境セットアップを実行してください")

setup_env = os.environ.copy()
setup_env["PYTHON"] = str(PI05_PYTHON)
setup_env["MUJOCO_GL"] = "egl"
setup_env["MPLBACKEND"] = "Agg"
subprocess.run(["bash", "setup.sh"], cwd=REPO_DIR, env=setup_env, check=True)

EVAL_PYTHON = REPO_DIR / "venv" / "bin" / "python"
if not EVAL_PYTHON.is_file():
    raise FileNotFoundError(EVAL_PYTHON)
print("評価Python:", EVAL_PYTHON)


### π0.5サーバーを起動して4タスクを実行

サーバーの起動確認後に4タスクを順番に評価し、成功率、collision rate、平均episode時間を表示します。各 `/act` と `/reset` には本番同様10秒の上限が適用されます。結果JSONとサーバーログは `results/pi05_public_eval/` に保存します。


In [ ]:
import json
import socket
import time
import urllib.request

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    SERVER_PORT = port_probe.getsockname()[1]
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
RESULTS_DIR = REPO_DIR / "results" / "pi05_public_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
server_log_path = RESULTS_DIR / "policy_server.log"
evaluation_log_path = RESULTS_DIR / "evaluation.log"
public_eval_result_path = RESULTS_DIR / f"server_{SERVER_PORT}.json"
if public_eval_result_path.exists():
    public_eval_result_path.unlink()

server_env = os.environ.copy()
server_env.update(
    {
        "PI05_MODEL_DIR": str(REPO_DIR / "submission_template/model_weights/pi05_libero_finetuned_v044"),
        "PI05_TOKENIZER_DIR": str(REPO_DIR / "submission_template/model_weights/paligemma-3b-pt-224"),
        "PI05_DEVICE": "cuda",
        "PI05_REPLAN_STEPS": "5",
        "PI05_INFERENCE_STEPS": "10",
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TOKENIZERS_PARALLELISM": "false",
    }
)
server_env["PYTHONPATH"] = os.pathsep.join(
    [
        "/content/pi05_runtime/lerobot_v044/src",
        "/content/pi05_runtime/transformers_lerobot_openpi/src",
        server_env.get("PYTHONPATH", ""),
    ]
)

eval_env = os.environ.copy()
eval_env.update(
    {
        "MUJOCO_GL": "egl",
        "MPLBACKEND": "Agg",
        "LIBERO_ROOT": str(REPO_DIR / "LIBERO-plus"),
        "PYTHONUNBUFFERED": "1",
    }
)
eval_env["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR / "LIBERO-plus"), str(REPO_DIR), str(REPO_DIR / "compe")]
)

server_command = [
    str(PI05_PYTHON),
    "policy_server.py",
    "--port",
    str(SERVER_PORT),
]
eval_command = [
    str(EVAL_PYTHON),
    "-m",
    "pipeline",
    "--server-url",
    SERVER_URL,
    "--track",
    "track1",
    "--n-episodes",
    str(EVAL_EPISODES_PER_TASK),
    "--max-steps",
    str(EVAL_MAX_STEPS),
    "--seed",
    str(EVAL_SEED),
    "--timeout",
    "10",
    "--output-dir",
    str(RESULTS_DIR),
    *(["--record-video"] if RECORD_VIDEO else []),
    "--videos-per-task",
    str(VIDEOS_PER_TASK),
    "--video-fps",
    "20",
    "--tasks",
    *PUBLIC_TASK_IDS,
]

evaluation_process = None
with server_log_path.open("w", encoding="utf-8") as server_log:
    server_process = subprocess.Popen(
        server_command,
        cwd=REPO_DIR / "submission_template",
        env=server_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    try:
        deadline = time.monotonic() + 180
        while time.monotonic() < deadline:
            if server_process.poll() is not None:
                raise RuntimeError(f"policy server exited early; see {server_log_path}")
            try:
                with urllib.request.urlopen(f"{SERVER_URL}/health", timeout=2) as response:
                    if response.status == 200:
                        break
            except Exception:
                time.sleep(1)
        else:
            raise TimeoutError(f"policy server did not start; see {server_log_path}")

        print("π0.5 policy server ready. 公開4タスク評価を開始します。", flush=True)
        evaluation_process = subprocess.Popen(
            eval_command,
            cwd=REPO_DIR,
            env=eval_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        recent_eval_lines = []
        assert evaluation_process.stdout is not None
        with evaluation_log_path.open("w", encoding="utf-8") as evaluation_log:
            for line in evaluation_process.stdout:
                print(line, end="", flush=True)
                evaluation_log.write(line)
                evaluation_log.flush()
                recent_eval_lines.append(line.rstrip())
                if len(recent_eval_lines) > 120:
                    recent_eval_lines.pop(0)
        evaluation_returncode = evaluation_process.wait()
        if evaluation_returncode != 0:
            raise RuntimeError(
                f"evaluation failed with exit={evaluation_returncode}\n"
                + "\n".join(recent_eval_lines)
            )
    finally:
        if evaluation_process is not None and evaluation_process.poll() is None:
            evaluation_process.terminate()
            try:
                evaluation_process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                evaluation_process.kill()
                evaluation_process.wait(timeout=5)
        server_process.terminate()
        try:
            server_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait(timeout=5)

if not public_eval_result_path.is_file():
    raise FileNotFoundError(public_eval_result_path)
public_eval_result = json.loads(public_eval_result_path.read_text(encoding="utf-8"))
track_result = public_eval_result["tracks"][0]
if track_result.get("overall_metrics", {}).get("error"):
    raise RuntimeError(f"track evaluation failed; see {public_eval_result_path}")
if len(track_result["tasks"]) != 4:
    raise RuntimeError(f"expected 4 task results, got {len(track_result['tasks'])}")

print("\n公開4タスク評価結果")
print("task | success | collision | avg episode sec")
print("--- | ---: | ---: | ---:")
for task in track_result["tasks"]:
    metrics = task["metrics"]
    print(
        f"{task['task_name']} | {task['success_rate']:.1%} | "
        f"{metrics.get('collision_rate', float('nan')):.1%} | "
        f"{metrics.get('avg_episode_time_sec', float('nan')):.1f}"
    )
print(f"overall success: {track_result['overall_score']:.1%}")
print("result JSON:", public_eval_result_path)
print("server log:", server_log_path)
print("evaluation log:", evaluation_log_path)
video_paths = sorted((RESULTS_DIR / "videos").glob("*.mp4"))
print(f"videos: {len(video_paths)}")
for video_path in video_paths:
    print("  ", video_path)


### 評価動画を確認

各タスクの先頭エピソードを、左にagent view、右にwrist viewを並べたMP4で表示します。20 episodes/taskでも既定では4本だけ保存します。


In [ ]:
from IPython.display import Video, display

video_paths = sorted(
    (REPO_DIR / "results" / "pi05_public_eval" / "videos").glob("*.mp4")
)
if not video_paths:
    print("動画がありません。RECORD_VIDEO=Trueで評価を実行してください。")
for video_path in video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=True, html_attributes="controls loop"))


In [ ]:
from google.colab import files

submission = REPO_DIR / "pi05_submission.zip"
if not submission.is_file():
    raise FileNotFoundError(submission)
print("submission size GiB:", round(submission.stat().st_size / 2**30, 2))
files.download(str(submission))
eval_result_candidates = sorted(
    (REPO_DIR / "results" / "pi05_public_eval").glob("server_*.json")
)
if eval_result_candidates:
    files.download(str(eval_result_candidates[-1]))


## 提出前の最終確認

このノートブックでは静的検査、単体推論、公開4タスク評価を行います。公開評価では各HTTPリクエストの10秒制限、成功率、collision rateを確認できます。提出前には完成ZIPを本番相当環境でも `python validate_submission.py pi05_submission.zip` で確認し、必要に応じて20 episodes/taskで再評価してください。
